# RUL Training (C-MAPSS FD001) — Li et al. (2018) Deep CNN

Faithful reimplementation of the RUL model from:
**Li, Ding, Sun (2018)**, *"Remaining useful life estimation in prognostics using deep
convolution neural networks"*, Reliability Engineering & System Safety 172, 1-11.

This notebook mirrors **`scripts/train_rul.py`**, the canonical trainer that produced the
shipped model (`models/rul_cnn.pt`, test RMSE 17.5). The script is the source of truth;
this notebook exists to make each stage readable and runnable cell by cell.

### Where each part lives in the repo

| Notebook section | Shipped file |
|---|---|
| 2. Preprocessing (features, RUL target, windows) | `scripts/train_rul.py` |
| 3. Network definition (`LiCNN`) | `src/rul_model.py` |
| 4. Training loop + evaluation | `scripts/train_rul.py` |
| Inference on live sensor data | `src/prediction.py` |
| Decision from the RUL estimate | `src/fusion.py` |

### Data

Needs the free NASA C-MAPSS set — only **FD001**: `train_FD001.txt`, `RUL_FD001.txt`,
and (for the RMSE report) `test_FD001.txt`. Set `CMAPSS_DIR` below.


## 1. Setup

Point `CMAPSS_DIR` at the folder holding the FD001 text files.

In [ ]:
!pip install -q torch numpy pandas

import json
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

CMAPSS_DIR = Path("CMAPSSData")     # <- folder with train_FD001.txt / RUL_FD001.txt
OUT_DIR    = Path("models"); OUT_DIR.mkdir(exist_ok=True)

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
print("FD001 files found:", sorted(p.name for p in CMAPSS_DIR.glob("*FD001*")))


## 2. Preprocessing

FD001 runs at a single operating condition, so 7 of the 21 sensors are flat and carry no
degradation signal — Li et al. keep 14; the 3 operational settings are kept as well, giving the
**17 features** the shipped model expects.

The RUL target is piecewise-linear: degradation is only observable once a unit is within
`R_EARLY` (125) cycles of failure, so the label is capped there. Features are min-max scaled
to [-1, 1] using **training statistics only**, and the same statistics are saved for
inference (`models/normalization_stats.json`) so training and deployment scale identically.
**Source — Li, Ding & Sun (2018)**, *Reliability Engineering & System Safety* 172, 1–11:

| What this implements | Where the paper states it |
|---|---|
| 14 of 21 sensors kept — the other 7 are constant over an engine's life and carry no RUL information | §3.1.2 *Data pre-processing*, p. 4 |
| Min-max scaling to [-1, 1], `2(x - x_min)/(x_max - x_min) - 1` | §3.1.2, p. 4 (Eq. 4) |
| Piecewise-linear RUL label, capped at `R_early` | §3.1.2, p. 4; value `R_early = 125` in Table 2, p. 6 |
| Time window of 30 cycles for FD001 | §3.1.3 *Time window processing*, p. 5; `N_tw = 30/20/30/15` in Table 2, p. 6 |

> Note: the paper uses `N_ft = 14` (sensors only). This repo feeds **17** features — the same
> 14 sensors plus the 3 operational settings — so the deviation is deliberate and explicit.


In [ ]:
# --- Feature selection (why): FD001 runs at a single operating condition, so 7 of the
# 21 sensors are flat and carry no degradation signal. Li et al. keep 14 informative
# sensors. The 3 operational settings are ALSO kept (17 features total) to match the
# model this repo ships; op_setting_3 is constant and normalizes to 0. ---
OP_SETTINGS = [1, 2, 3]
SENSORS = [2, 3, 4, 7, 8, 9, 11, 12, 13, 14, 15, 17, 20, 21]  # Li et al. FD001 set
FEATURE_COLUMNS = [f"op_setting_{i}" for i in OP_SETTINGS] + [f"sensor_{i}" for i in SENSORS]

WINDOW = 30       # Li et al. Ntw for FD001
R_EARLY = 125     # piecewise-linear RUL cap
COL_NAMES = ["unit", "cycle"] + [f"op_setting_{i}" for i in (1, 2, 3)] + [f"sensor_{i}" for i in range(1, 22)]


In [ ]:
def load_fd001(cmapss_dir: Path):
    """Read the whitespace-delimited C-MAPSS files into DataFrames."""
    import pandas as pd
    train = pd.read_csv(cmapss_dir / "train_FD001.txt", sep=r"\s+", header=None, names=COL_NAMES)
    rul_true = np.loadtxt(cmapss_dir / "RUL_FD001.txt")
    test_path = cmapss_dir / "test_FD001.txt"
    test = pd.read_csv(test_path, sep=r"\s+", header=None, names=COL_NAMES) if test_path.exists() else None
    return train, test, rul_true


def add_rul(df):
    """Piecewise-linear RUL target: (max_cycle_of_unit - cycle), capped at R_EARLY."""
    max_cycle = df.groupby("unit")["cycle"].transform("max")
    df = df.copy()
    df["RUL"] = np.minimum(max_cycle - df["cycle"], R_EARLY)
    return df


def fit_minmax(train_df):
    """Min-max stats on the training features only (no leakage from test)."""
    fmin = {c: float(train_df[c].min()) for c in FEATURE_COLUMNS}
    fmax = {c: float(train_df[c].max()) for c in FEATURE_COLUMNS}
    return {"feature_min": fmin, "feature_max": fmax}


def normalize(df, stats):
    """Map each feature to [-1, 1]; constant features (min==max) → 0."""
    out = df.copy()
    for c in FEATURE_COLUMNS:
        lo, hi = stats["feature_min"][c], stats["feature_max"][c]
        out[c] = 0.0 if hi == lo else 2 * (df[c] - lo) / (hi - lo) - 1
    return out


def make_train_windows(df):
    """Sliding windows of length WINDOW per engine unit; label = RUL at window end."""
    X, y = [], []
    for _, g in df.groupby("unit"):
        feats = g[FEATURE_COLUMNS].to_numpy(dtype="float32")
        ruls = g["RUL"].to_numpy(dtype="float32")
        if len(feats) < WINDOW:                       # short unit: left-pad with first row
            pad = np.repeat(feats[:1], WINDOW - len(feats), axis=0)
            feats = np.vstack([pad, feats]); ruls = np.concatenate([[ruls[0]], ruls])
        for s in range(len(feats) - WINDOW + 1):
            X.append(feats[s:s + WINDOW]); y.append(ruls[s + WINDOW - 1])
    return np.asarray(X, "float32"), np.asarray(y, "float32")


def make_test_windows(df):
    """Last WINDOW cycles per unit — the input Li et al. score against RUL_FD001.txt."""
    X = []
    for _, g in df.groupby("unit"):
        feats = g[FEATURE_COLUMNS].to_numpy(dtype="float32")
        if len(feats) < WINDOW:
            feats = np.vstack([np.repeat(feats[:1], WINDOW - len(feats), axis=0), feats])
        X.append(feats[-WINDOW:])
    return np.asarray(X, "float32")


## 3. Network — Li et al. (2018), Section 2.3

Four convolution layers of 10 filters, length 10, then one convolution with a single filter
of length 3, all tanh-activated; the map is flattened, dropped out (0.5), and passed through
a 100-unit fully-connected tanh layer to a single linear RUL output.

This is the exact class shipped as **`src/rul_model.py`** — the trainer and the inference
interface both import it, so a PyTorch `state_dict` always loads into a matching network.
**Source — Li et al. (2018), §2.3 *Proposed network structure*, p. 3**, with the values in
**Table 2, p. 6**:

| Hyperparameter | Paper (Table 2) | Here |
|---|---|---|
| Convolution layers | 5 | 5 (4 feature + 1 combining) |
| Filters per layer `F_N` | 10 | 10 |
| Filter length `F_L` | 10 | 10 |
| Neurons in fully-connected layer | 100 | 100 |
| Dropout rate | 0.5 | 0.5 |

> The final convolution uses a single filter of length 3 to combine the feature maps
> (§2.3, p. 3). Activations are tanh.


In [ ]:
import torch
import torch.nn as nn


class LiCNN(nn.Module):
    """Li et al. (2018) 1-D CNN RUL regressor."""

    def __init__(self, n_features: int = 17, window: int = 30):
        super().__init__()
        # Convolution runs along the time axis. PyTorch Conv1d expects
        # (batch, channels, length) = (batch, features, window), so forward() transposes.
        self.features = nn.Sequential(
            nn.Conv1d(n_features, 10, kernel_size=10, padding="same"), nn.Tanh(),
            nn.Conv1d(10, 10, kernel_size=10, padding="same"), nn.Tanh(),
            nn.Conv1d(10, 10, kernel_size=10, padding="same"), nn.Tanh(),
            nn.Conv1d(10, 10, kernel_size=10, padding="same"), nn.Tanh(),
            nn.Conv1d(10, 1, kernel_size=3, padding="same"), nn.Tanh(),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(window, 100), nn.Tanh(),
            nn.Linear(100, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, window, features) -> (batch, features, window)
        x = x.transpose(1, 2)
        return self.head(self.features(x)).squeeze(-1)   # (batch,)


## 4. Train

Paper settings: Adam, 250 epochs, batch 512, learning rate 1e-3 for the first 200 epochs
then 1e-4. On CPU this takes roughly 5-10 minutes.
**Source — Li et al. (2018), Table 2, p. 6** and **§3.1.5 *Prognostic procedure*, p. 5**:

| Setting | Paper | Here |
|---|---|---|
| Batch size | 512 (Table 2) | 512 |
| Epochs | 250 (Table 2) | 250 |
| Optimizer | Adam (§3.1.5, p. 5) | Adam |
| Learning rate | 0.001 for the first 200 epochs, then 0.0001 (§3.1.5, p. 5) | same |


In [ ]:
train_df, test_df, rul_true = load_fd001(CMAPSS_DIR)
train_df = add_rul(train_df)

stats = fit_minmax(train_df)
Xtr, ytr = make_train_windows(normalize(train_df, stats))
print("train windows:", Xtr.shape, " (features =", len(FEATURE_COLUMNS), ")")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = LiCNN(n_features=Xtr.shape[2], window=Xtr.shape[1]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()
loader = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)),
                    batch_size=512, shuffle=True)

EPOCHS = 250
for epoch in range(EPOCHS):
    for g in opt.param_groups:
        g["lr"] = 1e-3 if epoch < 200 else 1e-4
    model.train(); running = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item() * len(xb)
    if epoch % 25 == 0 or epoch == EPOCHS - 1:
        print(f"  epoch {epoch:3d}  mse={running / len(Xtr):.2f}")


## 5. Evaluate

Scored against the official `RUL_FD001.txt` labels, using the last 30 cycles of each test
unit — the protocol Li et al. report against (paper: RMSE ~12.6; this untuned run: ~17.5).

In [ ]:
Xte = make_test_windows(normalize(test_df, stats))
model.eval()
with torch.no_grad():
    pred = model(torch.from_numpy(Xte).to(device)).cpu().numpy()
pred = np.clip(pred, 0, None)

rmse = float(np.sqrt(np.mean((pred - rul_true) ** 2)))
print(f"TEST RMSE = {rmse:.2f}   (Li et al. FD001 ~ 12.6)")

for i in range(5):
    print(f"  unit {i}: predicted {pred[i]:6.1f}  |  true {rul_true[i]:5.0f}")


## 6. Save

Writes the two artifacts `src/prediction.py` loads at inference: the weights and the
normalization statistics. Copy them into the repo's `models/` to use them in the pipeline.

In [ ]:
torch.save(model.state_dict(), OUT_DIR / "rul_cnn.pt")
(OUT_DIR / "normalization_stats.json").write_text(json.dumps(stats, indent=2))
print("saved:", OUT_DIR / "rul_cnn.pt", "and", OUT_DIR / "normalization_stats.json")
